In [20]:
import sys
!{sys.executable} -m pip install prefect


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from prefect import flow, task
import pandas as pd
import numpy as np
import unicodedata
import warnings
import os

In [22]:
INVENTARIO    = r"C:\Users\Molly\Downloads\GMT-FT-110-01  INVENTARIO Y CRONOGRAMA 2026.xlsx"
HOJA          = 'INVENTARIO Y CRONOGRAMA'
ACTIVOS_FIJOS = r"C:\Users\Molly\Desktop\UAO\CLASE ETL\PROYECTO\Proyecto_final\Activos_Fijos_Data_ETL.xlsx"

In [23]:
@task
def extract_inventario():
    inv_df = pd.read_excel(INVENTARIO, sheet_name=HOJA, header=0)
    print(f"Inventario extraído: {inv_df.shape[0]} filas x {inv_df.shape[1]} columnas")
    return inv_df

@task
def extract_activos_fijos():
    act_fij = pd.read_excel(ACTIVOS_FIJOS, header=0)
    print(f"Activos fijos extraídos: {act_fij.shape[0]} filas x {act_fij.shape[1]} columnas")
    return act_fij

# ── TRANSFORM INVENTARIO ──────────────────────────────────────
@task
def transform_inventario(inv_df):
    NOMBRES_INV = ['ID','EQUIPO','MARCA','MODELO','SERIE','ACTIVO_FIJO','AREA','UBICACION',
                   'REGISTRO_SANITARIO','CLASIFICACION_RIESGO','ESTADO','PROPIETARIO_EQUIPO',
                   'ENCARGADO_MTTO_MP','PERIODICIDAD_MP']

    df = inv_df.iloc[5:, :14].copy()
    df.columns = NOMBRES_INV
    df = df.dropna(how='all').reset_index(drop=True)
    print(f"Filas extraídas: {len(df)}")

    # 1. Limpieza general
    def limpiar(v):
        if pd.isna(v): return np.nan
        v = str(v).replace('\n',' ').replace('\r',' ').replace('\t',' ')
        return ' '.join(v.split()).strip() or np.nan

    cols_inv = [c for c in df.columns if c != 'ID']
    for col in cols_inv:
        df[col] = df[col].apply(limpiar)
    print("✅ Limpieza de texto aplicada")

    # 2. Reconstruir ID
    df['ID_ORIGINAL'] = df['ID'].copy()
    df['ID_NUM']      = pd.to_numeric(df['ID'], errors='coerce')
    df = df.sort_values('ID_NUM', na_position='last').reset_index(drop=True)
    df['ID'] = range(1, len(df)+1)
    df = df.drop(columns=['ID_NUM'])
    print(f"✅ ID reconstruido: 1 → {len(df)}")

    # 3. Normalizar marcas y equipos
    mapa_marcas = {
        'abba':'ABBA','ABBA':'ABBA','Abbot':'Abbott',
        'air imean':'Air Imetan','AIR IMETAN':'Air Imetan','air imetan':'Air Imetan','Imetan':'Air Imetan',
        'baxter':'Baxter','BAXTER':'Baxter',
        'biocare':'BioCare','Biocare':'BioCare','BIOMERIEUX':'bioMerieux',
        'bionet':'Bionet','BIORAD':'Bio-Rad','BIORED':'Biored','BOECO':'Boeco',
        'BRAND':'Brand','BRIXCO':'Brixco','CONMED':'Conmed',
        'covidien':'Covidien','COVIDIEN':'Covidien',
        'Datex ohmeda':'Datex Ohmeda','datex ohmeda':'Datex Ohmeda',
        'EDAN':'Edan',
        'Fhiser and Payker':'Fisher & Paykel','Fisher & paykel':'Fisher & Paykel',
        'Fisher and Paykel health Care':'Fisher & Paykel',
        'GEneral electric':'General Electric','General electric':'General Electric','GE':'General Electric',
        'Generica':'Generico','generico':'Generico',
        'GENTEC':'Gentec','gentec':'Gentec',
        'Glucolquick':'GlucoQuick','Glucoquick':'GlucoQuick','GLUCOQUICK':'GlucoQuick',
        'GMD':'GMD','gmd':'GMD','HACEB':'Haceb','HANIL':'Hanil',
        'Hill room':'Hillrom','HillRom':'Hillrom','HILLROM':'Hillrom','HOMELife':'HomeLife',
        'Kex germani':'Kex Germany','KEX GERMANY':'Kex Germany','kex germany':'Kex Germany',
        'Kex germany':'Kex Germany','kex Germany':'Kex Germany','Germany':'Kex Germany',
        'KTJ':'KTJ','Ktj':'KTJ','ktj':'KTJ','ktj thermo':'KTJ',
        'L&k':'L&K','l&k':'L&K','LITMANN':'Littmann','littman':'Littmann',
        'LW SCIENTIFIC':'LW Scientific','LW SCINTIFIC':'LW Scientific',
        'mafet':'Mafet','mb':'MB','MEDIC':'Medic','MEMMERT':'Memmert',
        'MIDEA':'Midea','MINDRAY':'Mindray','mindray':'Mindray','mubi':'Mubi',
        'n.i':'No especifica','N.I':'No especifica','N.I.':'No especifica',
        'n.i.':'No especifica','N.I.0':'No especifica','NI':'No especifica',
        'OHAUS':'Ohaus','ohaus':'Ohaus','Ohio':'Ohio','ohio':'Ohio','OHIO':'Ohio',
        'OIxIgen':'Oxigen','IOxygen':'Oxigen','oxygen':'Oxigen',
        'OLYMPUS':'Olympus','omron':'Omron','OMRON':'Omron',
        'Precision medical':'Precision Medical','ROCHE':'Roche','riester':'Riester',
        'seca':'Seca','SECA':'Seca','schuco':'Schuco','SIEMENS':'Siemens',
        'steris':'Steris','STRYKER':'Stryker','sysmex':'Sysmex',
        'TENSO MED':'Tenso Med',
        'TERMO SCIENTIFIC':'Thermo Fisher Scientific',
        'THERMO FISHER SCIENTIFIC':'Thermo Fisher Scientific',
        'TERUMO':'Terumo','timeter':'Timeter','vacutron':'Vacutron',
        'Welch allyn':'Welch Allyn','WELCH ALLYN':'Welch Allyn','Welchallyn':'Welch Allyn',
        'zoll':'Zoll','ZOLL':'Zoll',
    }
    mapa_equipos = {
        'Agitador de MAZINI':'Agitador de Mazini',
        'Analizador de electrolitos':'Analizador de Electrolitos',
        'Analizador de gases':'Analizador de Gases',
        'Analizador de inmunohematologia':'Analizador de Inmunohematologia',
        'Analizador de inmunologia':'Analizador de Inmunologia',
        'Analizador de sangre':'Analizador de Sangre',
        'Bascula analoga':'Bascula Analoga','Bascula digital':'Bascula Digital',
        'Basculas analoga pediatrica':'Bascula Analoga Pediatrica',
        'Basculas digital pediatrica':'Bascula Digital Pediatrica',
        'Bano serologico':'Bano Serologico','Baño serologico':'Bano Serologico',
        'Bicicleta ergonomica':'Bicicleta Ergonomica',
        'Bomba de infusion':'Bomba de Infusion','Bomba de infusión':'Bomba de Infusion',
        'CaBina de Seguridad Biologica':'Cabina de Seguridad Biologica',
        'Cama hospitalaria':'Cama Hospitalaria',
        'centrifuga':'Centrifuga','consola quirurgica':'Consola Quirurgica',
        'ElectroBisturi':'Electrobisturi',
        'ESTERILIZADOR A GAS':'Esterilizador a Gas','EstroBoscopio':'Estroboscopio',
        'FONENDOSCOPIO':'Fonendoscopio','gh,.':'Sin Clasificar',
        'HemogloBinometro':'Hemoglobinometro','HUB RACK':'Hub Rack',
        'Lampara cuello de cisne':'Lampara Cuello de Cisne',
        'laringoscopio':'Laringoscopio','Lasser':'Laser',
        'Maquina de Aferesis':'Maquina de Aferesis',
        'Maquina de dialisis peritonial':'Maquina de Dialisis Peritoneal',
        'Maquina de hemodialisis':'Maquina de Hemodialisis',
        'Nevera de trasnporte conTermometro Digital':'Nevera de Transporte con Termometro Digital',
        'Nevera de transporte con termometro':'Nevera de Transporte con Termometro Digital',
        'Planta de osmosis portatil':'Planta de Osmosis Portatil',
        'Refrigerador de GloBulos Rojos':'Refrigerador de Globulos Rojos',
        'Regulador de oxigeno':'Regulador de Oxigeno',
        'Regulador de vacio':'Regulador de Vacio',
        'Sistema de deteccion MicroBiana':'Sistema de Deteccion Microbiana',
        'tallimetro':'Tallimetro','termohigrometro':'Termohigrometro',
        'termometro con sonda':'Termometro con Sonda','Termometro con sonda':'Termometro con Sonda',
        'Termometro digital':'Termometro Digital','Termometro Infrarojo':'Termometro Infrarrojo',
        'vaporizador':'Vaporizador','Videodudenoscopio':'Videoduodenoscopio',
        'Tensiometro pediatrico':'Tensiometro Pediatrico',
    }
    df['MARCA']  = df['MARCA'].replace(mapa_marcas)
    df['EQUIPO'] = df['EQUIPO'].replace(mapa_equipos)
    print(f"MARCA únicos: {df['MARCA'].nunique()} | EQUIPO únicos: {df['EQUIPO'].nunique()}")

    # 4. Normalizar categóricas
    df['ESTADO'] = df['ESTADO'].str.upper().str.strip()

    mapa_per = {
        'Anual':'ANUAL','anual':'ANUAL','ANUALES':'ANUAL','Anuales':'ANUAL','Anual ':'ANUAL',
        'Semestral':'SEMESTRAL','semestral':'SEMESTRAL',
        'Trimestral':'TRIMESTRAL','Cuatrimestral':'CUATRIMESTRAL','Bianual':'BIANUAL',
        'N.A':'N/A','N.A.':'N/A','N.a':'N/A',
    }
    df['PERIODICIDAD_MP'] = df['PERIODICIDAD_MP'].replace(mapa_per)

    mapa_riesgo = {
        'I':'CLASE I','CLASE I':'CLASE I','Clase I':'CLASE I',
        'IIA':'CLASE IIA','CLASE IIA':'CLASE IIA','Clase IIA':'CLASE IIA','Clase IIa':'CLASE IIA',
        'IIB':'CLASE IIB','CLASE IIB':'CLASE IIB',
        'N.A':'No Aplica','N.A.':'No Aplica',' ':np.nan,
    }
    df['CLASIFICACION_RIESGO'] = df['CLASIFICACION_RIESGO'].replace(mapa_riesgo)

    # 5. Eliminar tildes
    def sin_tildes(v):
        if pd.isna(v): return np.nan
        return ''.join(c for c in unicodedata.normalize('NFD', str(v))
                       if unicodedata.category(c) != 'Mn')

    cols_inv = [c for c in df.columns if c != 'ID']
    for col in cols_inv:
        df[col] = df[col].apply(sin_tildes)
    print("✅ Tildes eliminadas")

    # 6. Normalizar AREA
    mapa_area = {
        'Mto Biomedico':'Biomedico','MTO BIOMEDICO':'Biomedico','mto biomedico':'Biomedico','BIOMEDICO':'Biomedico',
        'Cardiologia no invasiva':'Cardiologia No Invasiva',
        'Central De Esterilizacion':'Central de Esterilizacion',
        'CENTRAL DE GASES':'Central de Gases','Central de gases':'Central de Gases',
        'Cirugia':'Cirugia','CIRUGIA':'Cirugia','CIrugia':'Cirugia','cirugia':'Cirugia',
        'Colsulta Especialista':'Consulta Especialista','COLSULTA ESPECIALISTA':'Consulta Especialista',
        'Hospitalizacion':'Hospitalizacion','HOSPITALIZACION':'Hospitalizacion',
        'Imagenes':'Imagenes Diagnosticas','Imagenes diagnosticas':'Imagenes Diagnosticas',
        'Oncologia':'Oncologia','oncologia':'Oncologia',
        'Rehabilitacion cardiaca':'Rehabilitacion Cardiaca',
        'Uci':'UCI','Uci 1':'UCI 1','Uci 2':'UCI 2',
        'Uci adultos':'UCI Adultos','Uci Adultos':'UCI Adultos',
        'Uci pediatrica':'UCI Pediatrica','uci pediatrica':'UCI Pediatrica',
        'Ucin':'UCIN','Ucis':'UCIS',
        'Unidad Transfucional':'Unidad Transfusional','unidad transfucional':'Unidad Transfusional',
        'Urgencias':'Urgencias','URGENCIAS':'Urgencias','urgencias':'Urgencias',
        'Terapia respiratoria':'Terapia Respiratoria',
        'Banco de sangre':'Banco de Sangre','Unidad renal':'Unidad Renal',
        'ARCHIVO':'Archivo','CMB':'CMB',
    }
    df['AREA'] = df['AREA'].replace(mapa_area)

    mapa_uci = {'UCI':'UCI 1', 'UCI Adultos':'UCI 2', 'UCIS':'UCI 2'}
    df['AREA'] = df['AREA'].replace(mapa_uci)

    mapa_prop = {'baxter':'Baxter','BAXTER':'Baxter','TERUMO':'Terumo','BIOART':'Bioart','proveedor':'Proveedor'}
    df['PROPIETARIO_EQUIPO'] = df['PROPIETARIO_EQUIPO'].replace(mapa_prop)

    mapa_enc = {'N.A':'N/A','N.A.':'N/A','Clase Iia':'No especifica'}
    df['ENCARGADO_MTTO_MP'] = df['ENCARGADO_MTTO_MP'].replace(mapa_enc)

    # 7. Imputar nulos
    reglas = {
        'AREA':'Biomedico','ESTADO':'Mantenimiento',
        'MARCA':'No especifica','MODELO':'No especifica','SERIE':'No especifica',
        'ACTIVO_FIJO':'No especifica','UBICACION':'No especifica',
        'REGISTRO_SANITARIO':'No especifica','CLASIFICACION_RIESGO':'No especifica',
        'PROPIETARIO_EQUIPO':'No especifica','ENCARGADO_MTTO_MP':'No especifica',
        'PERIODICIDAD_MP':'No especifica',
    }
    for col, val in reglas.items():
        df[col] = df[col].fillna(val)
    print("✅ Nulos imputados")

    # 8. Dimensión equipo
    dim = (df[['EQUIPO','MARCA','MODELO']].drop_duplicates()
           .sort_values(['EQUIPO','MARCA','MODELO'], key=lambda col: col.astype(str))
           .reset_index(drop=True))
    dim.insert(0, 'ID_EQUIPO', range(1, len(dim)+1))

    df = df.merge(dim, on=['EQUIPO','MARCA','MODELO'], how='left')

    ORDEN = ['ID','ID_ORIGINAL','ID_EQUIPO','EQUIPO','MARCA','MODELO','SERIE',
             'ACTIVO_FIJO','AREA','UBICACION','REGISTRO_SANITARIO','CLASIFICACION_RIESGO',
             'ESTADO','PROPIETARIO_EQUIPO','ENCARGADO_MTTO_MP','PERIODICIDAD_MP']
    df = df[ORDEN]

    print(f"✅ dim_equipo: {len(dim)} combinaciones únicas")
    print(f"   Inventario: {len(df)} filas x {len(df.columns)} columnas")
    return df, dim

# ── TRANSFORM ACTIVOS FIJOS ───────────────────────────────────
@task
def transform_activos_fijos(act_fij, mapa_marcas=None, mapa_equipos=None):
    # 1. Estandarizar nombres de columnas
    act_fij.columns = (act_fij.columns
                       .str.strip()
                       .str.upper()
                       .str.replace(' ', '_')
                       .str.replace('.', '', regex=False)
                       .str.replace('%', 'PCT')
                       .str.replace('Ó', 'O')   # ← agregar
                       .str.replace('Á', 'A')   # ← agregar
                       .str.replace('Ú', 'U'))  # ← agregar
    print(f"Columnas: {act_fij.columns.tolist()}")

    # 2. Limpieza general
    def limpiar(v):
        if pd.isna(v): return np.nan
        v = str(v).replace('\n',' ').replace('\r',' ').replace('\t',' ')
        return ' '.join(v.split()).strip() or np.nan

    cols_act = [c for c in act_fij.columns if c != 'ID']
    for col in cols_act:
        act_fij[col] = act_fij[col].apply(limpiar)
    print("✅ Limpieza de texto aplicada")

    # 3. Eliminar duplicados
    n_antes = len(act_fij)
    act_fij = act_fij.drop_duplicates(subset=["ID"]).reset_index(drop=True)
    print(f"✅ Duplicados eliminados: {n_antes - len(act_fij)}")

    # 4. Normalizar marcas y equipos
    mapa_marcas = mapa_marcas or {
        'abba':'ABBA','ABBA':'ABBA','baxter':'Baxter','BAXTER':'Baxter',
        'BIOMERIEUX':'bioMerieux','MINDRAY':'Mindray','mindray':'Mindray',
        'n.i':'No especifica','N.I':'No especifica','N.I.':'No especifica',
        'ROCHE':'Roche','SIEMENS':'Siemens','STRYKER':'Stryker',
        'Welch allyn':'Welch Allyn','WELCH ALLYN':'Welch Allyn',
        'zoll':'Zoll','ZOLL':'Zoll',
    }
    mapa_equipos = mapa_equipos or {
        'Agitador de MAZINI':'Agitador de Mazini',
        'Analizador de electrolitos':'Analizador de Electrolitos',
        'Bomba de infusión':'Bomba de Infusion',
        'Maquina de hemodialisis':'Maquina de Hemodialisis',
    }
    act_fij['MARCA']  = act_fij['MARCA'].replace(mapa_marcas)
    act_fij['EQUIPO'] = act_fij['EQUIPO'].replace(mapa_equipos)
    print(f"MARCA únicos: {act_fij['MARCA'].nunique()} | EQUIPO únicos: {act_fij['EQUIPO'].nunique()}")

    # 5. Eliminar tildes en valores
    def sin_tildes(v):
        if pd.isna(v): return np.nan
        return ''.join(c for c in unicodedata.normalize('NFD', str(v))
                       if unicodedata.category(c) != 'Mn')

    for col in cols_act:
        if col in act_fij.columns:
            act_fij[col] = act_fij[col].apply(sin_tildes)
    print("✅ Tildes eliminadas")

    # 6. Crear dim y merge
    dim_act = (act_fij[['EQUIPO','MARCA','MODELO']].drop_duplicates()
               .sort_values(['EQUIPO','MARCA','MODELO'], key=lambda col: col.astype(str))
               .reset_index(drop=True))
    dim_act.insert(0, 'ID_EQUIPO', range(1, len(dim_act)+1))
    act_fij = act_fij.merge(dim_act, on=['EQUIPO','MARCA','MODELO'], how='left')
    print(f"Columnas tras merge: {act_fij.columns.tolist()}")

    # 7. Seleccionar columnas para Power BI
    COLUMNAS_ACT = ['ID','ID_EQUIPO','COSTO','FECHA_INGRESO','FECHA_INICIO_OP',
                    'PRECIO_DEPRECIACION','AÑOS_USO','VIDA_UTIL',
                    'OBSOLESCENCIA_PCT','NIVEL_OBSOLESCENCIA','MTO_CORRECTIVO']
    data_activos = act_fij[COLUMNAS_ACT].copy()
    print(f'✅ Activos: {len(data_activos)} filas x {len(data_activos.columns)} columnas')
    return data_activos

# ── LOAD ─────────────────────────────────────────────────────
@task
def load_data(df, dim, data_activos):
    df.to_csv('inventario_estandarizado.csv', index=False, encoding='utf-8-sig', sep=';')
    dim.to_csv('dim_equipo.csv', index=False, encoding='utf-8-sig', sep=';')
    data_activos.to_csv('activos_fijos_data.csv', index=False, encoding='utf-8-sig', sep=';')

    print("=" * 55)
    for f in ['inventario_estandarizado.csv', 'dim_equipo.csv', 'activos_fijos_data.csv']:
        kb = os.path.getsize(f) / 1024
        print(f"✅ {f}  →  {kb:.1f} KB")
    print()
    print("⚠️  En Power BI: seleccionar 'Punto y coma' como delimitador al importar")
    print("=" * 55)
    return "Carga completada"

In [24]:
@flow(name="ETL_Inventario_Biomedico")
def etl_pipeline():
    # Extract
    inv_df  = extract_inventario()
    act_fij = extract_activos_fijos()

    # Transform
    df, dim  = transform_inventario(inv_df)
    act_fij  = transform_activos_fijos(act_fij)

    # Load
    status = load_data(df, dim, act_fij)
    print(status)

# Ejecutar
etl_pipeline()

00:18:50.208 | INFO    | Flow run 'daring-limpet' - Beginning flow run 'daring-limpet' for flow 'ETL_Inventario_Biomedico'

00:18:50.210 | INFO    | Flow run 'daring-limpet' - View at http://127.0.0.1:4200/runs/flow-run/9d5aa42d-4c54-4ac5-b4b3-bbbdb082e348

Inventario extraído: 1710 filas x 67 columnas


c:\Users\Molly\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


00:18:51.311 | INFO    | Task run 'extract_inventario-6f0' - Finished in state Completed()

...

00:18:51.769 | INFO    | Task run 'transform_activos_fijos-324' - Finished in state Completed()

✅ inventario_estandarizado.csv  →  264.6 KB
✅ dim_equipo.csv  →  23.3 KB
✅ activos_fijos_data.csv  →  131.4 KB

⚠️  En Power BI: seleccionar 'Punto y coma' como delimitador al importar


00:18:51.814 | INFO    | Task run 'load_data-2d6' - Finished in state Completed()

Carga completada


00:18:52.215 | INFO    | Flow run 'daring-limpet' - Finished in state Completed()